# LeCun Persona Fine-Tune — Colab Training Notebook

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU)

Stages run here:
1. Clone repo + install deps
2. Rebuild data splits (clean → chunk → build)
3. Train (QLoRA 4-bit on CUDA)
4. Save adapter to Google Drive

In [ ]:
# Verify we have a GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU, then reconnect.')
print(result.stdout)

In [ ]:
# Clone the repo (force-fresh so stale sessions always get latest code)
import shutil, os
if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')
%cd /content
!git clone https://github.com/VayunMalik/NLPFinalProject repo
%cd repo

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Rebuild data splits from raw sources
# (processed/ is gitignored so we regenerate here)
!python run_pipeline.py --stages clean chunk build

In [ ]:
# Log in to Hugging Face to download Qwen2.5-7B-Instruct
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# Train — full run with 4-bit QLoRA on CUDA
# Remove --max-steps to train for all 3 epochs (~20-40 min on T4)
!python run_pipeline.py --stages train

In [ ]:
# Mount Drive and copy the adapter out so it survives session end
from google.colab import drive
drive.mount('/content/drive')

import shutil, pathlib

src = pathlib.Path('model/adapters/run_default')
dst = pathlib.Path('/content/drive/MyDrive/lecun_adapter')
dst.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)
print(f'Adapter saved to {dst}')
print(list(dst.iterdir()))

In [ ]:
# Optional: quick inference check with the saved adapter
!python model/inference.py \
    --adapter model/adapters/run_default \
    --prompt "What is the key challenge in building safe AI systems?" \
    --system-prompt "You are Yann LeCun. Respond in his style." \
    --max-new-tokens 200